# 05 用 3Blue1Brown 视角重讲 Attention 与 Transformer

这一节先暂停继续推进新概念。

我们换一种讲法，把前面学过的内容重新串起来。

参考的是 3Blue1Brown 在深度学习系列里讲 Transformer 和 Attention 的思路。

他的讲法有一个很大的特点：

```text
不先背公式。
先看 token 的向量为什么需要被上下文改写。
再用一个具体故事解释 Q、K、V 各自的职责。
最后才把这些动作压缩成矩阵乘法。
```

这一节会比较详细。

目标是让你脑子里先出现画面，再回头看公式。

## 1. 先说参考来源

这一节参考了 3Blue1Brown 的两篇官方文字版课程：

- [Transformers, the tech behind LLMs | Deep Learning Chapter 5](https://www.3blue1brown.com/lessons/gpt/)
- [Attention in transformers, step-by-step | Deep Learning Chapter 6](https://www.3blue1brown.com/lessons/attention/)

这里不会照搬原文，而是把他的讲课顺序整理成适合我们当前笔记的版本。

你可以把这一节当成：

```text
把 00 到 04 的内容，换成 3Blue1Brown 的视角重新走一遍。
```

## 2. 3Blue1Brown 不是从 RNN 讲起

你前面问过：是不是因为没学 RNN，所以看不懂 Attention。

看 3Blue1Brown 的讲法，会发现他也不是先完整讲 RNN。

他更关心的是：

```text
Transformer 接收什么？
内部数据长什么样？
Attention 到底怎么让 token 向量互相传递信息？
```

也就是说，理解 Attention 的主线不是：

```text
先学完 RNN -> 再学 Attention
```

而是：

```text
先知道文本怎么变成 token
-> token 怎么变成向量
-> 向量为什么需要上下文
-> Attention 如何让向量互相传信息
```

## 3. Transformer 的目标：预测下一个 token

3Blue1Brown 先把 GPT 类模型的目标说得很简单。

输入一段文本，模型要预测后面可能接什么。

例如：

```text
今天 天气 很
```

模型可能预测：

```text
好
冷
热
糟糕
```

它不是直接输出一个确定答案，而是输出一个概率分布。

所以从大方向看，GPT 类 Transformer 的任务可以理解成：

```text
读入上下文，把上下文信息压进向量里，然后根据这些向量预测下一个 token。
```

## 4. 文本先被切成 token

模型不能直接吃一整段自然语言。

它要先把文本切成 token。

为了入门理解，可以先把 token 理解成词、字，或者词的一部分。

例如：

```text
小红 明天 要 考试
```

可以先看成：

```text
token 1：小红
token 2：明天
token 3：要
token 4：考试
```

如果有 $N$ 个 token，我们就说这段输入有 $N$ 个位置。

这一点和我们前面一直说的 `N` 是同一件事：

```text
N = token 数量 = 位置数量
```

## 5. token 再变成 embedding

token 还是文字，神经网络处理不了。

所以每个 token 会先被映射成一个向量。

这个向量通常叫 embedding。

例如：

```text
小红 -> [0.21, -0.15, 0.77, ...]
明天 -> [0.02, 0.43, -0.31, ...]
考试 -> [0.84, 0.12, 0.39, ...]
```

3Blue1Brown 很喜欢从几何角度理解 embedding。

可以想象每个词都变成高维空间里的一个点或一个方向。

相似的词，可能在这个空间里离得比较近。

某些方向可能对应某种含义，比如性别、国家、复数、职业、情绪等。

我们不需要把这些方向真的画出来，只要先接受一个直觉：

```text
embedding 是模型用一串数字表示 token 含义的方式。
```

## 6. 初始 embedding 的问题：它还没有上下文

这里是 3Blue1Brown 讲 Attention 的关键入口。

初始 embedding 通常像查表一样得到。

也就是说，同一个 token 刚进模型时，得到的是比较通用的表示。

比如英文里的 `mole` 可以表示：

```text
鼹鼠
化学里的摩尔
皮肤上的痣
```

但是刚开始查到的 `mole` embedding，本身还不知道它处在哪个句子里。

如果输入是：

```text
one mole of carbon dioxide
```

它应该偏向化学含义。

如果输入是：

```text
a mole on the skin
```

它应该偏向皮肤上的痣。

所以问题来了：

```text
模型怎么把一个通用的 token 向量，改写成符合当前上下文的向量？
```

## 7. Attention 的目标：改写 embedding

3Blue1Brown 对 Attention 的讲法，不是先说“它是一个加权求和公式”。

他先让你想象：

```text
每个 token 的向量一开始只知道自己。
经过 Attention Block 后，它要吸收上下文，变成更具体的新向量。
```

也就是说，Attention Block 的作用可以理解成：

```text
让 token 向量之间互相传递信息，并更新彼此的表示。
```

比如 `mole` 的向量，经过上下文影响后，可以被推向“化学摩尔”的方向，也可以被推向“皮肤痣”的方向。

这个说法比直接背 QKV 更重要。

因为 QKV 只是实现这个目标的工具。

## 8. 一个非常重要的画面：上下文把向量推到新方向

你可以把一个 token 的 embedding 想成空间里的一个箭头。

初始箭头代表这个词比较通用的含义。

Attention 的工作，是根据上下文给它加上一些调整量。

比如：

```text
原来的向量：tower 的通用含义
看到 Eiffel：往 Eiffel Tower / Paris / France 的方向调整
看到 miniature：再往“小型的”方向调整
```

这就是 3Blue1Brown 很强调的视觉化直觉：

```text
Attention 不是把词替换掉。
而是在高维空间里调整这个词向量的位置或方向。
```

换成我们前面的话：

```text
输入 x_i
经过 Attention
变成融合上下文后的新表示 c_i 或 o_i
```

## 9. 用“形容词更新名词”作为核心例子

3Blue1Brown 在讲 Attention 时，用了一个非常适合入门的例子。

假设有一句话：

```text
A fluffy blue creature roamed the forest.
```

我们暂时只关心一种更新：

```text
让形容词的信息更新对应名词的 embedding。
```

比如：

```text
fluffy 应该更新 creature
blue 也应该更新 creature
```

因为 creature 一开始只是“生物”这个通用概念。

但看到 fluffy 和 blue 后，它应该变成更具体的含义：

```text
一只毛茸茸的蓝色生物
```

这个例子好在它非常清楚地说明了 Attention 的目标：

```text
不是分类。
不是直接预测答案。
而是让某些词的信息流向另一些词，改写它们的向量表示。
```

## 10. Query：名词提出一个问题

在这个例子里，名词 creature 需要吸收形容词的信息。

那么 creature 可以像是在提出一个问题：

```text
我前面有没有形容词可以修饰我？
```

这个问题，就由 Query 向量来表示。

也就是：

$$
\mathbf{q}_{creature}
$$

注意，Query 不是一句人类语言问题。

它是一个向量。

但这个向量在模型里的功能，就像是在表达一种检索需求。

所以可以这样理解：

```text
Query = 当前 token 想从上下文里找什么信息。
```

## 11. Key：每个词提供匹配信号

Query 提出了问题。

那谁来回答这个问题？

句子里的每个 token 都会提供一个 Key。

例如：

```text
fluffy 有自己的 key
blue 有自己的 key
creature 有自己的 key
roamed 有自己的 key
forest 有自己的 key
```

这些 Key 可以理解成每个 token 对外展示的“匹配标签”。

当 creature 的 Query 去匹配所有 Key 时，模型会判断：

```text
fluffy 是否适合回答 creature 的问题？
blue 是否适合回答 creature 的问题？
roamed 是否适合回答 creature 的问题？
forest 是否适合回答 creature 的问题？
```

所以：

```text
Key = 每个 token 用来被别人匹配的表示。
```

## 12. 点积：Query 和 Key 怎么匹配

Query 和 Key 都是向量。

模型需要一种方法来判断两个向量是否匹配。

3Blue1Brown 在前一节专门铺垫过点积直觉：点积可以衡量两个方向有多对齐。

在 Attention 中，可以先这样写：

$$
s_{i,j}=\mathbf{q}_i\cdot\mathbf{k}_j
$$

读作：

```text
第 i 个 token 看第 j 个 token 的相关性分数。
```

如果 creature 的 Query 和 fluffy 的 Key 点积很大，就说明 fluffy 对更新 creature 很相关。

如果 creature 的 Query 和 roamed 的 Key 点积较小，就说明 roamed 在这个注意力头里可能不太相关。

这里说“在这个注意力头里”，是因为不同头可以学不同关系，后面讲 Multi-Head 时会展开。

## 13. 注意力图：所有 Query-Key 分数排成一张表

如果句子有 $N$ 个 token。

每个 token 都有 Query。

每个 token 也都有 Key。

每个 Query 都和每个 Key 算一个分数。

所以会得到一张 $N\times N$ 的表：

```text
             key 1   key 2   key 3   ...   key N
query 1       分数     分数     分数   ...    分数
query 2       分数     分数     分数   ...    分数
query 3       分数     分数     分数   ...    分数
...          ...      ...      ...    ...    ...
query N       分数     分数     分数   ...    分数
```

3Blue1Brown 把这类表可视化成 attention pattern。

我们可以把它叫做注意力图或注意力表。

这张图的含义非常直接：

```text
每一格表示一个 token 对另一个 token 的关注程度。
```

## 14. Softmax：把分数变成比例

点积得到的是原始分数。

这些分数还不能直接拿来加权求和。

因为它们可能是负数，也不一定加起来等于 1。

所以每一行要经过 Softmax。

为什么按行？

因为每一行表示：

```text
某一个 Query 看所有 Key 的分数。
```

这一行 Softmax 后，就变成：

```text
这个 token 应该从所有 token 那里各拿多少信息。
```

例如 creature 看所有词的权重可能是：

```text
fluffy：0.45
blue：0.35
creature：0.15
其他词：接近 0
```

这表示 creature 这个位置主要吸收 fluffy 和 blue 的信息。

## 15. Value：真正提供“怎么更新”的内容

现在有了注意力权重。

但权重本身只告诉我们：

```text
应该关注谁，以及关注多少。
```

它还没有告诉我们：

```text
被关注的词应该给目标词带来什么具体改变。
```

这就是 Value 的作用。

在形容词更新名词的例子里：

```text
fluffy 的 Value 可能携带“毛茸茸”这种调整信息。
blue 的 Value 可能携带“蓝色”这种调整信息。
```

creature 对 fluffy 的权重大，就会更多吸收 fluffy 的 Value。

creature 对 blue 的权重大，也会更多吸收 blue 的 Value。

所以：

```text
Key 决定谁相关。
Value 决定相关时要贡献什么内容。
```

## 16. 用一句话串起 Q、K、V

现在可以把 Q、K、V 重新串起来。

以 creature 为例：

```text
creature 的 Query 提问：谁能修饰我？
每个词的 Key 回答：我是否适合被你关注？
Query 和 Key 点积，得到相关性分数。
Softmax 把分数变成权重。
每个词的 Value 提供具体更新内容。
creature 按权重汇总这些 Value。
creature 的 embedding 被改写。
```

这就是 3Blue1Brown 讲 Attention 最值得借鉴的地方：

```text
QKV 不是三个缩写。
它们是一个信息流动故事里的三个角色。
```

## 17. 这和我们前面学的公式怎么对应

前面我们写过：

$$
s_{i,j}=\mathbf{q}_i\cdot\mathbf{k}_j
$$

这对应：

```text
第 i 个 token 的问题，和第 j 个 token 的匹配信号，对得上多少。
```

然后：

$$
\alpha_{i,j}=\operatorname{softmax}(s_{i,j})
$$

这对应：

```text
第 i 个 token 从第 j 个 token 那里拿多少信息。
```

最后：

$$
\mathbf{o}_i=\sum_j\alpha_{i,j}\mathbf{v}_j
$$

这对应：

```text
第 i 个 token 按权重汇总所有 token 提供的更新内容。
```

这三步就是 Attention 的灵魂。

## 18. 矩阵公式只是批量写法

如果每个 token 都这样做一遍，写起来会很啰嗦。

所以把所有 Query 放进矩阵 $Q$。

把所有 Key 放进矩阵 $K$。

把所有 Value 放进矩阵 $V$。

于是得到常见公式：

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

它可以读成：

```text
QK^T：所有 Query 和所有 Key 互相匹配，得到注意力分数表。
除以 sqrt(d_k)：缩放分数，让 Softmax 更稳定。
Softmax：把每一行分数变成权重。
乘 V：按权重汇总 Value，得到更新后的 token 表示。
```

所以矩阵公式不是另一套东西。

它只是把很多个“谁看谁”的小计算放在一起算。

## 19. 注意一个记号差异：QK^T 和 K^TQ

你看不同资料时，可能会看到不同写法。

我们前面的笔记采用常见的行向量习惯：

```text
Q：N x d_k
K：N x d_k
QK^T：N x N
```

有些可视化资料会把 token 向量画成列向量。

这样写出来可能会变成类似：

```text
K^TQ
```

这不是本质冲突。

只是向量摆放方向不同。

真正不变的是：

```text
每个 Query 都要和每个 Key 算匹配分数。
最后得到 token 和 token 之间的注意力表。
```

## 20. 为什么 3Blue1Brown 先讲一个 attention head

他不是一开始就讲 Multi-Head。

而是先讲 single head，也就是一个注意力头。

原因很简单：

```text
一个 head 可以先学一种关系。
```

比如这个 head 暂时专门学：

```text
形容词如何更新名词。
```

另一个 head 以后可能学：

```text
代词如何找到指代对象。
动词如何关联主语。
时间词如何影响事件理解。
否定词如何影响句子含义。
```

这也解释了为什么 Multi-Head Attention 不是“把一个东西重复很多遍凑参数”。

更好的直觉是：

```text
多个 head 并行，让模型从不同角度建模上下文关系。
```

## 21. 为什么 Transformer 要堆很多 Attention Block

3Blue1Brown 在补充文章里提到过一个很有启发的想法：

前面的 Attention Block 可能先让某些词吸收局部信息。

后面的 Attention Block 再基于已经更新过的向量，做更复杂的关系判断。

举个中文例子：

```text
玻璃球掉在钢桌上，它碎了。
```

如果一开始只看“球”和“桌”，模型不一定知道“它”更可能指谁。

但如果前面的层已经让“球”吸收了“玻璃”的信息，让“桌”吸收了“钢”的信息，那么后面的层就更容易判断：

```text
会碎的更可能是玻璃球，而不是钢桌。
```

这说明 Transformer 不是只做一次 Attention 就完事。

它会一层一层改写 token 向量。

浅层可能学比较直接的关系。

深层可能基于前面已经整理过的信息，学更抽象的关系。

## 22. Attention Block 和 MLP Block 的分工

在 3Blue1Brown 的整体介绍里，Transformer 里反复交替出现两类模块：

```text
Attention Block
MLP / Feed-Forward Block
```

可以先这样理解：

```text
Attention Block：让不同 token 的向量互相交流。
MLP Block：每个 token 向量各自内部加工。
```

也就是说，Attention 更像是在回答：

```text
我应该从上下文里的哪些 token 那里拿信息？
```

MLP 更像是在回答：

```text
拿到信息后，我这个 token 自己内部还要做哪些特征变换？
```

这个对比对后面理解 Transformer Encoder 很有帮助。

## 23. 为什么 GPT 里要 mask

3Blue1Brown 讲 GPT 时还会提到 mask。

GPT 类模型是预测下一个 token。

训练时，第 1 个位置不能偷看第 2 个位置之后的答案。

第 2 个位置不能偷看第 3 个位置之后的答案。

所以在注意力表里，后面的 token 不能影响前面的 token。

这就需要 mask。

简单理解：

```text
允许当前位置看自己和前面。
不允许当前位置看未来。
```

在数学上，常见做法是在 Softmax 前，把不允许看的位置设成非常小的数，接近负无穷。

这样经过 Softmax 后，这些位置的权重就会变成 0。

这一节只先知道 GPT 里有这个限制，不展开代码。

## 24. 3Blue1Brown 的讲法和我们笔记的区别

我们前面的笔记更像课程讲义：

```text
定义概念
解释 QKV
推形状
写公式
做总结
```

3Blue1Brown 的讲法更像视觉故事：

```text
先问 token embedding 为什么不够用
再看上下文如何改写 embedding
再用具体关系解释一个 attention head
最后把动作压缩成矩阵
```

这两种讲法不是冲突的。

它们适合解决不同问题。

如果你觉得公式很抽象，就用 3Blue1Brown 的故事去理解。

如果你想写代码和检查维度，就回到我们第 04 节的形状路线。

## 25. 把 00 到 04 重新串成一条线

现在把前面所有内容按 3Blue1Brown 的风格重排：

```text
文本被切成 token。
每个 token 变成 embedding。
初始 embedding 只知道自己，还不懂上下文。
Attention Block 负责让 token 向量互相交流。
一个 token 的 Query 表示它想找什么。
每个 token 的 Key 表示它如何被匹配。
Query 和 Key 点积形成注意力分数表。
Softmax 把分数表变成注意力权重表。
每个 token 的 Value 表示它能贡献什么更新内容。
权重乘 V 后，每个 token 得到融合上下文的新表示。
很多层反复做这件事，最后某些向量里包含足够多上下文信息，可以用于预测下一个 token。
```

这就是 Attention 在 Transformer 里的核心位置。

## 26. 用我们自己的中文例子再走一遍

看这句话：

```text
聪明 勤奋 的 学生 通过了 考试
```

假设一个 attention head 学的是：

```text
让形容词更新名词。
```

那么“学生”这个 token 的 Query 可能像是在问：

```text
有哪些词在描述我？
```

“聪明”和“勤奋”的 Key 可能很匹配这个问题。

所以它们获得较大权重。

然后“聪明”和“勤奋”的 Value 会被更多汇总进“学生”的新表示。

更新前：

```text
学生：比较通用的学生概念
```

更新后：

```text
学生：聪明、勤奋、通过考试的学生
```

这就是 Attention 让 token 表示变具体的过程。

## 27. 再用代词指代例子走一遍

看这句话：

```text
小明把复习资料交给小红，因为她明天要考试。
```

假设一个 attention head 学的是：

```text
让代词找到它指代的人。
```

那么“她”的 Query 可能像是在问：

```text
我指的是前面哪个人物？
```

“小明”和“小红”都会提供 Key。

但结合“明天要考试”“交给”等上下文，小红的 Key 可能更匹配。

于是“她”对“小红”的注意力权重更大。

最后“小红”的 Value 更多汇入“她”的新表示。

更新后，“她”的向量就更像是在表示：

```text
小红这个人，且和明天考试相关。
```

## 28. 你现在应该怎么读 QKV 公式

以后再看到：

$$
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

不要第一反应去背。

你可以这样读：

```text
Q：每个 token 带着自己的问题。
K：每个 token 提供可匹配的信号。
QK^T：所有问题和所有匹配信号互相打分。
除以 sqrt(d_k)：把分数缩放一下。
Softmax：把分数变成每个 token 的关注比例。
V：每个 token 提供真正要被拿走的内容。
乘 V：每个 token 根据关注比例吸收上下文，得到新向量。
```

如果这段能读顺，公式就已经被你驯服了一半。

## 29. 这一节最重要的三层理解

第一层：故事层。

```text
token 向量需要吸收上下文，变得更具体。
```

第二层：机制层。

```text
Query 提问，Key 匹配，Value 提供更新内容。
```

第三层：矩阵层。

```text
QK^T 生成注意力表，Softmax 变成权重，乘 V 得到新表示。
```

学习 Attention 时，最好不要只停在其中一层。

只看故事，会觉得懂了但写不出公式。

只看公式，会觉得每个字母都冰冷。

把三层互相对应起来，才是真正理解。

## 30. 本节小结

这一节按 3Blue1Brown 的视角，把 Attention 和 Transformer 重新串了一遍。

重点记住：

1. Transformer 处理的是 token 向量。
2. 初始 embedding 通常还没有上下文。
3. Attention Block 的目标是让 token 向量互相传递信息。
4. 一个 token 的向量会被上下文推向更具体的方向。
5. Query 表示当前 token 想找什么。
6. Key 表示每个 token 如何被匹配。
7. Query 和 Key 的点积形成注意力分数。
8. Softmax 把分数变成权重。
9. Value 表示真正贡献给对方的更新内容。
10. 权重乘 V 后，每个 token 得到融合上下文的新表示。
11. Multi-Head 可以理解成多个不同关系的 attention head 并行工作。
12. 多层 Transformer 会反复改写 token 向量，让它们逐渐携带更丰富的上下文信息。

## 31. 自测问题

1. 3Blue1Brown 为什么先讲 token 和 embedding，而不是先讲 QKV 公式？
2. 为什么初始 embedding 还不能充分表示上下文含义？
3. Attention Block 的目标可以怎样理解？
4. 用 `mole` 一词多义的例子说明为什么 embedding 需要被上下文改写。
5. 在“形容词更新名词”的例子里，名词的 Query 可以理解成什么问题？
6. Key 在这个例子里提供什么作用？
7. Value 和 Key 为什么不是一回事？
8. 为什么 Query 和 Key 的点积可以形成注意力分数？
9. 注意力图或 attention pattern 的每一格表示什么？
10. Softmax 在注意力图里起什么作用？
11. 为什么矩阵公式只是批量写法？
12. `QK^T` 和 `K^TQ` 的写法差异为什么不必恐慌？
13. Multi-Head Attention 为什么可以理解成多个关系角度并行？
14. 为什么 Transformer 要堆很多 Attention Block 和 MLP Block？
15. GPT 里的 mask 主要是为了防止什么？